In [1]:
import sys
import os
sys.path += [ f'{os.environ["HOME"]}/.local/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages' ]

# Now we can safely import atlasopenmagic
import atlasopenmagic as atom

In [2]:
import uproot # for reading .root files
import time # to measure time to analyse
import math # for mathematical functions such as square root
import awkward as ak # for handling complex and nested data structures efficiently
import numpy as np # # for numerical calculations such as histogramming
import matplotlib.pyplot as plt # for plotting
from matplotlib.ticker import MaxNLocator,AutoMinorLocator # for minor ticks
from lmfit.models import PolynomialModel, GaussianModel # for the signal and background fits
import vector #to use vectors
import requests # for HTTP access
import aiohttp # HTTP client support
import pandas as pd
import json

In [3]:
atom.set_release('2025e-13tev-beta')

Fetching metadata for release: 2025e-13tev-beta...
Fetching datasets: 100%|██████████| 374/374 [00:00<00:00, 792.71datasets/s]
✓ Successfully cached 374 datasets.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


In [4]:
lumi = 36

In [5]:
# list with signal DSIDs: ee, mumu
# should tautau, bb, tt be included?
dsid_signal_list = [301209] 
dsid_background_list = [700323, 700324, 700325, 700470, 700471, 700472, 410219, 700589, 700602, 601624, 601628]
#dsid_background_list = [700324, 700325, 700470, 700471, 700472]

In [6]:
for dsid in dsid_background_list:
    print(dsid, atom.get_metadata(dsid, 'physics_short'))
    tree = uproot.open(atom.get_urls(dsid, protocol='root', cache=False)[0] + ":analysis")
    print("The information stored in the tree is:", tree.keys())

# are these corect?

700323 Sh_2211_Zmumu_maxHTpTV2_BFilter
The information stored in the tree is: ['num_events', 'sum_of_weights', 'sum_of_weights_squared', 'xsec', 'kfac', 'filteff', 'TriggerMatch_DILEPTON', 'ScaleFactor_MLTRIGGER', 'ScaleFactor_PILEUP', 'ScaleFactor_FTAG', 'mcWeight', 'channelNumber', 'eventNumber', 'runNumber', 'trigML', 'trigP', 'trigDT', 'trigT', 'trigE', 'trigDM', 'trigDE', 'trigM', 'trigMET', 'ScaleFactor_BTAG', 'ScaleFactor_JVT', 'jet_n', 'jet_pt', 'jet_eta', 'jet_phi', 'jet_e', 'jet_btag_quantile', 'jet_jvt', 'largeRJet_n', 'largeRJet_pt', 'largeRJet_eta', 'largeRJet_phi', 'largeRJet_e', 'largeRJet_m', 'largeRJet_D2', 'jet_pt_jer1', 'jet_pt_jer2', 'ScaleFactor_ELE', 'ScaleFactor_MUON', 'ScaleFactor_LepTRIGGER', 'ScaleFactor_MuTRIGGER', 'ScaleFactor_ElTRIGGER', 'lep_n', 'lep_type', 'lep_pt', 'lep_eta', 'lep_phi', 'lep_e', 'lep_charge', 'lep_ptvarcone30', 'lep_topoetcone20', 'lep_z0', 'lep_d0', 'lep_d0sig', 'lep_isTightID', 'lep_isMediumID', 'lep_isLooseID', 'lep_isTightIso', 'lep_

In [7]:
def get_xsec_weight(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
        / metadata["sumOfWeights"]
    )

def get_N_inclusive(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
    )

def get_inclusive_yield(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
    )

def calc_weight(xsec_weight, weight_arr, data):
    for variable in weight_arr:
        xsec_weight = xsec_weight * data[variable]
    return xsec_weight

In [8]:
# number of signal events, might be scaled up
metadata = atom.get_metadata(dsid_signal_list[0])
xsec_weight = get_xsec_weight(metadata, lumi)
signal_nevents = get_N_inclusive(metadata, lumi)
signal_nevents

63.784800000000004

In [9]:
background_yield_sum = 0
for dsid in dsid_background_list:
    metadata = atom.get_metadata(dsid)
    inc_yield = get_inclusive_yield(metadata, lumi)
    background_yield_sum = background_yield_sum + inc_yield

background_yield_sum

170264948.425092

In [10]:
def create_objects(data, variables):

    prefixes = {
        "lep": "lep_",
        "jet": "jet_",
        "tau": "tau_",
        "photon": "photon_",
        "largeRJet": "largeRJet_",
        "ScaleFactor": "ScaleFactor_",
        "trig": "trig_",
        "met": "met_"}

    objects = {}

    for variable in variables:
        # special case: met
        if variable == "met":
            if "met" not in objects:
                objects["met"] = {}
            objects["met"]["met"] = data[variable]
            continue

        found = False

        for category, prefix in prefixes.items():
            if variable.startswith(prefix):
                name = variable[len(prefix):]
                if category not in objects:
                    objects[category] = {}
                objects[category][name] = data[variable]
                found = True
                break

        if not found:
            if "info" not in objects:
                objects["info"] = {}
            objects["info"][variable] = data[variable]

    # zip everything
    for category in objects:
        objects[category] = ak.zip(objects[category])

    return objects

In [11]:
def create_blackbox(signal_dsid_list, background_dsid_list, variables, signal_nevents, background_nevents, lumi=36, 
                    skim="noskim", checkpoint_dir="blackbox_checkpoint",):
    
    print("Variables:", variables)

    # create checkpoint directories
    events_dir = os.path.join(checkpoint_dir, "events")
    completed_file = os.path.join(checkpoint_dir, "completed_dsids.json")

    os.makedirs(events_dir, exist_ok=True)


    # load checkpoint information
    if os.path.exists(completed_file):

        print(f"Load checkpoint: {checkpoint_dir}")

        with open(completed_file, "r") as f:
            completed_dsids = set(json.load(f))

        print(f"{len(completed_dsids)} DSIDs already processed.")
        print(f"Completed DSIDs: {completed_dsids}")

    else:
        print("No checkpoint found. Start from scratch.")
        completed_dsids = set()


    # calculate inclusive yields
    yield_dict = {}

    signal_yield_sum = 0
    background_yield_sum = 0

    for dsid in signal_dsid_list:

        metadata = atom.get_metadata(dsid)
        inc_yield = get_inclusive_yield(metadata, lumi)

        yield_dict[dsid] = inc_yield
        signal_yield_sum += inc_yield


    for dsid in background_dsid_list:

        metadata = atom.get_metadata(dsid)
        inc_yield = get_inclusive_yield(metadata, lumi)

        yield_dict[dsid] = inc_yield
        background_yield_sum += inc_yield


    # calculate number of events per DSID
    nevents_per_sample_dict = {}

    for dsid in signal_dsid_list:
        nevents_per_sample = (signal_nevents / signal_yield_sum * yield_dict[dsid])
        nevents_per_sample_dict[dsid] = nevents_per_sample


    for dsid in background_dsid_list:
        nevents_per_sample = (background_nevents / background_yield_sum * yield_dict[dsid])
        nevents_per_sample_dict[dsid] = nevents_per_sample


    # save completed DSIDs
    def save_completed_dsids():
        with open(completed_file, "w") as f:
            json.dump(sorted(completed_dsids), f, indent=2)
        print(f"Checkpoint information saved: " f"{len(completed_dsids)} completed DSIDs")


    # process samples
    dsid_labels = ([(dsid, 1) for dsid in signal_dsid_list] + [(dsid, 0) for dsid in background_dsid_list])

    for dsid, label in dsid_labels:

        dsid_str = str(dsid)

        print("\n" + "=" * 60)
        print(f"Current DSID: {dsid}")
        print(f"Label: {label}")
        print(f"Already processed: {dsid_str in completed_dsids}")

        # skip already completed DSID
        if dsid_str in completed_dsids:
            print(f"DSID {dsid} already processed -> SKIP")
            continue

        # number of events to collect
        target_events = int(nevents_per_sample_dict[dsid])

        remaining = target_events
        collected_events = 0

        print(f"Target events: {target_events}")

        # temporary storage for this DSID
        chunks = {}
        label_chunks = []

        # get ROOT files
        file_list = atom.get_urls(dsid, skim, protocol="root", cache=False)
        print("Number of files:", len(file_list))

        # loop over ROOT files
        for file_number, afile in enumerate(file_list, start=1):

            print(f"\nProcessing file " f"{file_number}/{len(file_list)}")

            if collected_events >= target_events:
                break

            # read ROOT file chunk-by-chunk
            for data in uproot.iterate(afile + ":analysis", variables, library="ak", step_size=100_000,):

                if len(data) == 0:
                    continue

                # only take events that are still needed
                if len(data) > remaining:
                    data = data[:remaining]

                n_events = len(data)

                # create objects from variable names
                new_objects = create_objects(data, variables)
                label_chunks.append(ak.Array([label] * n_events))

                # save in chunks
                for name, obj in new_objects.items():
                    if name not in chunks:
                        chunks[name] = []
                    chunks[name].append(obj)

                # update counters
                collected_events += n_events
                remaining = target_events - collected_events

                print(f"Collected events: " f"{collected_events}/{target_events}")

                if collected_events >= target_events:
                    break

        # check whether enough events were found
        if collected_events < target_events:
            print(f"WARNING: Only found " f"{collected_events}/{target_events} events " f"for DSID {dsid}.")

        # combine chunks for this DSID
        dsid_arrays = {}

        for name, chunk_list in chunks.items():
            if len(chunk_list) > 0:
                dsid_arrays[name] = ak.concatenate(chunk_list, axis=0)

        # combine labels
        if len(label_chunks) == 0:
            print(f"WARNING: No events collected for DSID {dsid} -> SKIP")
            continue
        dsid_arrays["label"] = ak.concatenate(label_chunks, axis=0)

        # create one Awkward record for this DSID
        dsid_data = ak.zip(dsid_arrays, depth_limit=1)

        # save DSID checkpoint
        checkpoint_file = os.path.join(events_dir, f"dsid_{dsid_str}.parquet")

        print(f"Saving {len(dsid_data)} events to " f"{checkpoint_file}")

        ak.to_parquet(dsid_data, checkpoint_file)

        # mark DSID as completed
        completed_dsids.add(dsid_str)

        save_completed_dsids()


    # load all checkpoint files
    print("\nLoading all checkpoint files...")

    parquet_files = [os.path.join(events_dir, f) for f in os.listdir(events_dir) if f.endswith(".parquet")]

    parquet_files.sort()

    if len(parquet_files) == 0:
        raise RuntimeError("No events found in checkpoint.")

    for parquet_file in parquet_files:
        ds = ak.from_parquet(parquet_file)

        print(parquet_file)
        print("type:", ak.type(ds))
        print("label type:", ak.type(ds["label"]))
        print("label:", ds["label"][:10])

    # load and concatenate
    data = ak.from_parquet(parquet_files)

    print(f"Loaded {len(data)} events " f"from {len(parquet_files)} parquet files.")


    # shuffle
    rng = np.random.default_rng(25)
    indices = rng.permutation(len(data))
    data = data[indices]

    # labels
    labels = data["label"]

    return data, labels

In [12]:
data, labels = create_blackbox(dsid_signal_list, dsid_background_list, ["lep_pt", "jet_e", "lep_isTightID", "xsec"], 10, 100, skim="2muons")

Variables: ['lep_pt', 'jet_e', 'lep_isTightID', 'xsec']
Load checkpoint: blackbox_checkpoint
6 DSIDs already processed.
Completed DSIDs: {'700472', '700325', '700471', '700323', '700324', '301209'}

Current DSID: 301209
Label: 1
Already processed: True
DSID 301209 already processed -> SKIP

Current DSID: 700323
Label: 0
Already processed: True
DSID 700323 already processed -> SKIP

Current DSID: 700324
Label: 0
Already processed: True
DSID 700324 already processed -> SKIP

Current DSID: 700325
Label: 0
Already processed: True
DSID 700325 already processed -> SKIP

Current DSID: 700470
Label: 0
Already processed: False
Target events: 0
Number of files: 1

Processing file 1/1

Current DSID: 700471
Label: 0
Already processed: True
DSID 700471 already processed -> SKIP

Current DSID: 700472
Label: 0
Already processed: True
DSID 700472 already processed -> SKIP

Current DSID: 410219
Label: 0
Already processed: False
Target events: 0
Number of files: 1

Processing file 1/1

Current DSID: 700

In [13]:
data

<Array [{lep: [...], jet: [...], ...}, ...] type='107 * {lep: var * {pt: fl...'>

In [14]:
data["lep"]["isTightID"]

<Array [[True, True], [True, ...], ..., [True, True]] type='107 * var * bool'>

In [15]:
labels

<Array [0, 0, 0, 0, 0, 1, 0, 0, ..., 1, 0, 0, 0, 0, 0, 0, 0] type='107 * int64'>